<a href="https://colab.research.google.com/github/zahariesergiu/ubb-sociology-ml/blob/ionela-tugui/Examen_Tugui_Ionela.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import re
import nltk

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec

nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [2]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 66.3 MB/s eta 0:00:00


In [4]:
df = pd.read_csv('set_date_examen1.csv')

df.head()

,rest_id,text,rating
0,L772e6l2Yd0DJEyCBxBNng,Everytime I'm in town I hit this place for cur...,5.0
1,L772e6l2Yd0DJEyCBxBNng,I go to State Street Brats every time I am in ...,4.0
2,L772e6l2Yd0DJEyCBxBNng,Two of us visited on a Thursday lunch.\n\nThe ...,3.0
3,L772e6l2Yd0DJEyCBxBNng,State street brats has a dress code that openl...,1.0
4,L772e6l2Yd0DJEyCBxBNng,Sad to have a place like this on campus in Mad...,1.0


In [5]:
df = df[df['rating'] != 3]

def sentiment_label(r):
    if r <= 2:
        return 0
    else:
        return 1

df['sentiment'] = df['rating'].apply(sentiment_label)

df['sentiment'].value_counts()

,count
sentiment,
1,7756
0,2181


In [10]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)

    return text

In [11]:
X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [12]:
vectorizer = CountVectorizer(max_features=5000)

X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow = vectorizer.transform(X_test)

In [13]:
rf_bow = RandomForestClassifier(n_estimators=100, random_state=42)

rf_bow.fit(X_train_bow, y_train)

y_pred_bow = rf_bow.predict(X_test_bow)

In [16]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [17]:
print("BOW + Random Forest:\n")
print(classification_report(y_test, y_pred_bow))

BOW + Random Forest:

              precision    recall  f1-score   support

           0       0.95      0.62      0.75       436
           1       0.90      0.99      0.95      1552

    accuracy                           0.91      1988
   macro avg       0.93      0.81      0.85      1988
weighted avg       0.91      0.91      0.90      1988



Interpretare

Modelul BOW cu Random Forest a obtinut o acuratete ridicata 91%, ceea ce arata ca majoritatea recenziilor sunt clasificate corect.

Pentru clasa pozitiva, modelul are rezultate foarte bune, cu un recall de 0.99 si un f1-score de 0.95, ceea ce inseamna ca aproape toate recenziile pozitive sunt identificate corect.

Pentru clasa negativa, rezultatele sunt mai slabe, in special recall-ul de 0.62, ceea ce arata ca modelul nu detecteaza toate recenziile negative. Totusi, precision-ul este mare (0.95), deci atunci cand prezice negativ, de obicei are dreptate.

Se observa si faptul ca seturile nu sunt echilibrate, existand mult mai multe recenzii pozitive (1552) decat negative (436), ceea ce influenteaza performanta modelului si il face mai bun pe clasa majoritara.

In [18]:
X_train_tokens = X_train.apply(word_tokenize)
X_test_tokens = X_test.apply(word_tokenize)

In [19]:
w2v_model = Word2Vec(
    sentences=X_train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

In [20]:
def get_sentence_vector(tokens, model):
    vectors = []

    for word in tokens:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [21]:
X_train_w2v = np.array([
    get_sentence_vector(tokens, w2v_model)
    for tokens in X_train_tokens
])

X_test_w2v = np.array([
    get_sentence_vector(tokens, w2v_model)
    for tokens in X_test_tokens
])

In [22]:
rf_w2v = RandomForestClassifier(n_estimators=100, random_state=42)

rf_w2v.fit(X_train_w2v, y_train)

y_pred_w2v = rf_w2v.predict(X_test_w2v)

In [23]:
print("Word2Vec + Random Forest:\n")
print(classification_report(y_test, y_pred_w2v))

Word2Vec + Random Forest:

              precision    recall  f1-score   support

           0       0.81      0.58      0.68       436
           1       0.89      0.96      0.93      1552

    accuracy                           0.88      1988
   macro avg       0.85      0.77      0.80      1988
weighted avg       0.87      0.88      0.87      1988



Interpretare

Modelul Word2Vec cu Random Forest a obtinut o acuratete de 88%, ceea ce arata ca majoritatea recenziilor sunt clasificate corect.

Pentru clasa pozitiva, modelul are rezultate foarte bune, cu precision de 0.89, recall de 0.96 si f1-score de 0.93.

Pentru clasa negativa, performanta este mai scazuta, cu precision de 0.81, recall de 0.58 si f1-score de 0.68. Recall-ul mai mic arata ca modelul nu reuseste sa detecteze o parte importanta din recenziile negative.


(5p) Comparați cele doua modele și alegeți care este cel mai bun?

Comparand cele doua modele, se observa ca modelul BOW cu Random Forest obtine rezultate mai bune, avand o acuratete de 0.91 fata de 0.88 pentru Word2Vec.

De asemenea, modelul BOW are un f1-score mai ridicat pentru clasa pozitiva (0.95 fata de 0.93) si performante similare sau mai bune per total.

Ambele modele sunt influentate de dezechilibrul datelor (436 recenzii negative vs 1552 recenzii pozitive), ceea ce duce la rezultate mai bune pe clasa majoritara.

In plus, in etapa de preprocesare au fost eliminate recenziile neutre (rating = 3), transformand problema intr-o clasificare binara. Acest lucru a simplificat sarcina modelului si a contribuit la obtinerea unor rezultate mai bune. Totusi, chiar si dupa eliminarea acestora, setul de date a ramas dezechilibrat, ceea ce continua sa influenteze performanta modelelor.

In [25]:
#(10p) Creați o analiza de sentiment pe setul de date construind o arhitectura de rețele neuronale pentru clasificare.



In [26]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 10000
max_len = 100

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

In [27]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, GlobalAveragePooling1D

model = Sequential([
    Embedding(input_dim=max_words, output_dim=64, input_length=max_len),
    GlobalAveragePooling1D(),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [28]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/5
199/199 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.7809 - loss: 0.5114 - val_accuracy: 0.7887 - val_loss: 0.4514
Epoch 2/5
199/199 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8643 - loss: 0.3140 - val_accuracy: 0.9283 - val_loss: 0.2277
Epoch 3/5
199/199 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9393 - loss: 0.1686 - val_accuracy: 0.9377 - val_loss: 0.1734
Epoch 4/5
199/199 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9630 - loss: 0.1109 - val_accuracy: 0.9314 - val_loss: 0.1710
Epoch 5/5
199/199 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9720 - loss: 0.0857 - val_accuracy: 0.9396 - val_loss: 0.1472


In [29]:
y_pred_nn = (model.predict(X_test_pad) > 0.5).astype("int32")

print("Neural Network:\n")
print(classification_report(y_test, y_pred_nn))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Neural Network:

              precision    recall  f1-score   support

           0       0.89      0.83      0.86       436
           1       0.95      0.97      0.96      1552

    accuracy                           0.94      1988
   macro avg       0.92      0.90      0.91      1988
weighted avg       0.94      0.94      0.94      1988

